<a href="https://colab.research.google.com/github/marcory-hub/Seeed_Grove_Vision_AI_Module_V2/blob/main/YOLO11n_training_2026_02_19.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# YOLO11n training




Last accessed: 2026-06-07 runtime 2026.04 T4 GPU
- add version freeze
- train with cv_vespa_2025-05v2_top (only vcra and vvel)

1. Make sure images and labels from your dataset have this folder structure with these exact names. And add `data.yaml` to main folder.

```
🗂️ dataset
  🗂️ train
    🗂️ images
    🗂️ labels
  🗂️ valid
    🗂️ images
    🗂️ labels
  data.yaml
```

2. Zip the dataset folder to a file names `dataset.zip`. On mac use
```
cd /PATH_TO/data
rm -f dataset.zip
find . -type f \( -name '*.jpg' -o -name '*.txt' -o -name '*.yaml' \) -print | zip -rX dataset.zip -@
```

3. Copy the `dataset.zip` file to /`content/drive/MyDrive`, it is needed to make a callibration image set and your yolo model, fe `best.pt` to this folder. For the model you can use a custom name and adjust it in the options below.

4. Copy the `dataset.zip` file to the folder /content/drive/MyDrive/yolo.


In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#check GPU
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

Mon Jun  8 17:06:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Adjust the zipped file name

In [ ]:
# Copy zipped dataset to colab and unzip the dataset
!cp '/content/drive/MyDrive/dataset.zip' '/content/dataset.zip'
!unzip '/content/dataset.zip' -d '/content/dataset'

Streaming output truncated to the last 5000 lines.
  inflating: /content/dataset/train/labels/vcra2_1876_jpg.rf.5f7ab93f7908313148a3824cb8ac5246.txt  
  inflating: /content/dataset/train/labels/vvel2_6026_jpg.rf.c5ae466d7d60bc08185d145941a7bc46.txt  
  inflating: /content/dataset/train/labels/vvel1_1376_jpg.rf.6a1469dc67466c94426da1184461acdb.txt  
  inflating: /content/dataset/train/labels/vvel2_1077_jpg.rf.59a56ce6ebc9990b815916154d9472b9.txt  
  inflating: /content/dataset/train/labels/vvel2_2690_jpg.rf.6d90746b22b01de7dc95535292aec1b7.txt  
  inflating: /content/dataset/train/labels/vvel2_6031_jpg.rf.7f729d283f31fbf58c0f8517637fb274.txt  
  inflating: /content/dataset/train/labels/vcra2_5572_jpg.rf.15ab242f2c29d4be1cd9d8eb5f1e524c.txt  
  inflating: /content/dataset/train/labels/vvel2_5344_jpg.rf.a5fcf1778797f2903cc68756e0cbaef2.txt  
  inflating: /content/dataset/train/labels/vcra2_4058_jpg.rf.8154416ae6017c6c6312d4375fd3a17b.txt  
  inflating: /content/dataset/train/labels/vcra2_

In [ ]:
# Install the required packages for Ultralytics YOLO and Weights & Biases
!pip install -U ultralytics==8.4.14 wandb==0.27.0 numpy==2.0.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 64.4 MB/s eta 0:00:00


In [ ]:
from ultralytics import YOLO

# Initialize YOLO
yolo = YOLO()

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.



# Train, zip and download the model

Action:
1. Adjust epochs (select 10 for testing purposes, 300 for training)
2. Adust batch_size: common batch sizes that work well with GPU architectures are powers of two, such as 16, 32, 64, 128, ... 512. -1 for autobatch. (395 worked with T4-high ram)
3. Set image size (default 192, max 224).



-----

In [ ]:
import os
import shutil
from google.colab import userdata, drive
from ultralytics import YOLO, settings

drive_save_path = "/content/drive/MyDrive/YOLO_trainings"

# Enable W&B in Ultralytics
os.environ["WANDB_API_KEY"] = userdata.get('wandb-key')
settings.update({"wandb": True})

# Config
project_name = "vst_2026-05v2_top"
name = "yolo11n_2026-05v2_top_e300_b395_cls3"

# Train
model = YOLO("yolo11n.pt")

model.train(
    data="/content/dataset/data.yaml",
    epochs=300,
    patience=30,
    batch=395,
    imgsz=224,
    lr0=0.01,
    cls=3.0,
    project=project_name,
    name=name
)

# Backup to Drive
final_local_path = f"{project_name}/{name}"
final_drive_path = f"{drive_save_path}/{project_name}/{name}"

os.makedirs(os.path.dirname(final_drive_path), exist_ok=True)
if os.path.exists(final_local_path):
    shutil.copytree(final_local_path, final_drive_path, dirs_exist_ok=True)
    print(f"Results backed up to: {final_drive_path}")


New https://pypi.org/project/ultralytics/8.4.62 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.14 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=395, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=3.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo11n_2026-0

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: mvdijk (mvdijk-vespcv) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Overriding model.yaml nc=80 with nc=2

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      6640  ultralytics.nn.modules.block.C3k2            [32, 64, 1, False, 0.25]      
  3                  -1  1     36992  ultralytics.nn.modules.conv.Conv             [64, 64, 3, 2]                
  4                  -1  1     26080  ultralytics.nn.modules.block.C3k2            [64, 128, 1, False, 0.25]     
  5                  -1  1    147712  ultralytics.nn.modules.conv.Conv             [128, 128, 3, 2]              
  6                  -1  1     87040  ultralytics.nn.modules.block.C3k2            [128, 128, 1, True]           
  7                  -1  1    295424  ultralytics

In [ ]:
import shutil
from google.colab import files

# Paths
source_folder = "/content/runs/detect/vst_2026-05v2_top"
zip_file = "/content/vst_2026-05v2_top_e300_b395_cls3.zip"

# Create ZIP
shutil.make_archive(zip_file.replace('.zip',''), 'zip', source_folder)
print(f"✅ Folder zipped to {zip_file}")

# Download to local computer
files.download(zip_file)
